# LakeSignal Setup

Complete setup workflow for a new LakeSignal environment.

## Prerequisites

The UC **catalog** must already exist. You need permission to create a schema, volumes, and tables in it.

Set `CATALOG`, `SCHEMA`, `WHEEL_FILE`, and optionally `SERVICE_PRINCIPAL_ID` in the next cell, then run all cells in order.

In [ ]:
# UC identifiers (catalog must already exist)
CATALOG = "my_catalog"
SCHEMA = "lake_signal"

SERVICE_PRINCIPAL_ID = None  # e.g. UUID string for GRANTS; omit to skip

## Step 1: Bootstrap schema

Creates the schema and UC volume needed before uploading the wheel. Idempotent — safe to re-run.

In [ ]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS IDENTIFIER('{CATALOG}.{SCHEMA}')
    WITH DBPROPERTIES ('delta.enablePredictiveOptimization' = 'true')
""")

print(f"Schema {CATALOG}.{SCHEMA} and volume lakesignal ready")

## Step 2: Create tables via `setup()`

If the notebook cannot import `lakesignal` in the cell below immediately after pip, **restart Python** (`Cluster` > `Restart`) or `%restart_python`, then rerun from `from lakesignal import ...` onward.

`setup()` uses a `schema_version` table to track deployed DDL versions. On a fresh install it applies all migrations (creates tables, grants). On subsequent calls it skips already-applied versions and applies only incremental steps. Safe to re-run.

In [ ]:
from lakesignal import setup

# `spark` is provided in notebook / job compute
setup(
    spark,
    catalog=CATALOG,
    schema=SCHEMA,
    service_principal=SERVICE_PRINCIPAL_ID,
)

print(f"LakeSignal DDL ensured for {CATALOG}.{SCHEMA}")

In [ ]:
%skip   
# grant privs to users and groups
principal = ""
spark.sql(f"GRANT USE CATALOG ON CATALOG IDENTIFIER('{CATALOG}') TO `{principal}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA IDENTIFIER('{SCHEMA}') TO `{principal}`")
spark.sql(f"GRANT MODIFY, SELECT ON SCHEMA IDENTIFIER('{SCHEMA}') TO `{principal}`")
spark.sql(f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME IDENTIFIER('{CATALOG}.{SCHEMA}.LakeSignal') TO `{principal}`")

